# Lab 07: Walking Through an AI Coding Agent
### COMM 4259 <br> <i>Foundations of Machine Learning and AI with Python</i>

© Jingjing Li- Ask for permission for using/quoting: jingjing.li@virginia.edu

## Lab Prep
* If you would like to see my screen on your computer, you can use a web browser to log in to Zoom, turn off the speaker and mute yourself.
    - Section 001, 11:00 a.m. – 12:15 p.m.: https://mcintire.zoom.us/j/95993311893?pwd=NmabjMgwCFyKkxzib3qMFahEDkkaSI.1
    - Section 002, 2:00 – 3:15 p.m.: https://mcintire.zoom.us/j/97629389481?pwd=GA37uPdbfC0154k4M3n17SGf6qaYCg.1
* Have VS Code open with your `comm4259-work` folder before we start.

## Lab Rules
* Raise hands for questions
* Be prepared to share your screen for Q & A or exercises

# Learning Objectives
In this lab you work through the full workflow you will use in **Assignment 2, Section 2**, on different data (the Customer data from Lab 05/06). You will:
1. Make data-preparation decisions yourself and record the evidence for each one.
2. Turn your decisions into a **plan-mode** prompt and review the agent's plan before it writes code.
3. Understand what Streamlit is, and build and run a small Streamlit dashboard with Claude Code.
4. Use the agent's controls from Lab 04 (model, effort, permissions), watch its **context**, and give it standing course rules with `CLAUDE.md`.
5. Export the session and **verify** the agent's numbers with your own pandas code.
6. Ask the agent for a different option, compare the two with evidence, and choose one.

The agent writes code quickly, but the decisions and the checking are yours. Do not use the Titanic data in this lab; those decisions belong to your assignment.

# Before You Start
- Claude Code is installed and signed in with your UVA account (setup guide in the **Course Setup** module on Canvas).
- You unzipped `lab_07.zip` into `comm4259-work`, so you have `comm4259-work/lab_07/` with this notebook, `Customer.csv` and `Data_Dictionary.xlsx`.
- In VS Code, `comm4259-work` is open, and you have a terminal (Terminal > New Terminal).

**Working safely with an agent**
- Start Claude Code from your `comm4259-work` folder, so that it can see your course files and your `CLAUDE.md`. When it asks whether you trust the folder, answer Yes for `comm4259-work`; never trust your whole user folder.
- In each prompt, say which folder it may change (here `lab_07/agent_demo`). Your `CLAUDE.md` repeats this rule.
- Never paste passwords, API keys or personal data into the chat.
- Read every permission prompt before you approve it. Shift+Tab cycles through the permission modes; the bottom line of Claude Code shows which mode is on.

# Why Claude Code in the Terminal
Claude Code also comes as a panel inside VS Code (the Claude extension). In this course we run it in the terminal, for two reasons.

1. **The newest features arrive in the terminal first.** Claude Code is a command-line tool, and the VS Code panel is built on top of it. New commands and options usually reach the terminal version first, and the official documentation and most examples you will find online use the terminal.
2. **You choose the settings each time you start it.** The words you add after `claude` are called **flags** (or options). They set up the session before it begins:

| You type | What it does |
|---|---|
| `claude --continue` (or `claude -c`) | reopen your last conversation in this folder, if you closed it by accident |
| `claude --resume` | choose an earlier conversation from a list |
| `claude --model opus` | start with a particular model |
| `claude --permission-mode plan` | start in plan mode |
| `claude -p "your question"` | answer one question, print the answer and exit |
| `claude --help` | list every flag |

A flag starts with `--` (long form) or `-` (short form). Some flags are on/off switches, such as `--continue`. Others need a value right after them, such as `--model opus`. The order of the flags usually does not matter.

You can also save your own setup in the terminal's settings file, so that it works the same way every time. The optional section at the end of this lab adds a shortcut that chooses which Claude account to start with, and lists VS Code extensions that help you review the agent's work.

Two more advantages:
- Each terminal is its own Claude session, so you can run two tasks side by side (Terminal > Split Terminal) and give each terminal tab a name (right-click the tab > Rename).
- Inside Claude, type `/ide` to connect it to VS Code. The agent's edits then open as VS Code diffs, so you can review them line by line.

One flag to avoid: `--dangerously-skip-permissions` turns off every permission prompt for the whole session. Its name is the warning.

# Streamlit in Five Minutes
Before the agent builds anything, see what it will be building.

**Streamlit** is a Python library that turns a Python script into an interactive web page. You write ordinary Python (pandas, charts), add a few `st.` commands, and Streamlit draws the page in your browser.

- **How it runs.** `streamlit run app.py` starts a small web server on your own computer and opens the app in your browser (usually at `http://localhost:8501`). Only you can see it. To stop it, click in the terminal and press **Ctrl+C**.
- **The app keeps its terminal busy.** While the app runs, you cannot type other commands in that terminal. Open a **second terminal** with the **+** in the terminal panel and type your next commands there; the app keeps running in the first one. A new terminal starts in `comm4259-work`. When you no longer need the app, click in its terminal and press **Ctrl+C**.
- **How it updates.** Every time someone changes a widget (a dropdown, a slider), Streamlit runs the whole script again from the top and redraws the page.
- **The commands you will see most:**

| Command | What it shows |
|---|---|
| `st.title("...")`, `st.header("...")` | a title or a section heading |
| `st.write(...)` | text, numbers or a small table |
| `st.metric("label", value)` | one number in large type |
| `st.dataframe(df)` | a scrollable table |
| `st.selectbox("label", options)` | a dropdown; returns the user's choice |
| `st.bar_chart(series)` | a bar chart |
| `st.tabs(["A", "B"])` | tabs, one per part of the app |

First, check which version of Streamlit you have. It comes with Anaconda, so this cell should print a number. Write it down; Step 4 uses it. If you see `No module named 'streamlit'`, you are probably using the wrong kernel: choose **base (anaconda3)** from **Select Kernel** and run the cell again. If it still fails, ask a TA rather than installing packages during the lab.

In [48]:
import streamlit
print(streamlit.__version__)

1.58.0


### Your first Streamlit app (no AI)
Run the cell below. It writes a small app, `hello_app.py`, into your `lab_07` folder. Read each line: it reads the data, shows two numbers, and lets the user pick a column to see the churn rate by group.

In [49]:
%%writefile hello_app.py
from pathlib import Path
import pandas as pd
import streamlit as st

st.title("Customer churn: a first look")

customer = pd.read_csv(Path(__file__).parent / "Customer.csv")   # the data file next to this app

st.metric("Customers", len(customer))
st.metric("Churn rate", f"{customer['Churn'].mean():.1%}")

column = st.selectbox("Show the churn rate by", ["Gender", "Education", "Children", "State"])
st.bar_chart(customer.groupby(column)["Churn"].mean())

st.dataframe(customer.head(10))

Overwriting hello_app.py


Now run it. In the VS Code terminal, from `comm4259-work`:

```
streamlit run lab_07/hello_app.py
```

If `streamlit` is not recognized (common on Windows), use `python -m streamlit run lab_07/hello_app.py`. The app finds `Customer.csv` next to its own file, so it runs from any folder.

**The first time** you run Streamlit, it asks for an email address in the terminal ("If you'd like to receive helpful onboarding emails..."). Leave it blank and press **Enter**. It does not ask again.

The app opens in your browser. Change the dropdown and watch the chart change: that is the script running again from the top.

**Try it:** add `"State"` to the dropdown list, save the file, and click **Rerun** in the browser (top right). Then press **Ctrl+C** in the terminal to stop the app.

The dashboard the agent builds later works the same way, just with more of these commands. Because you know what each one does, you can read its code and check it.

# Step 1: Look at the Data Yourself (no AI)
Run the cell below. It uses the same commands as Lab 05/06 to show the size of the data, the number and share of missing values in each column, and the two formats of ZipCode.

In [50]:
import pandas as pd

customer = pd.read_csv("Customer.csv")
print(customer.shape)

# Number and share of missing values in each column
print(customer.isna().sum())
print(customer.isnull().mean())

# How many characters does each ZipCode have?
print(customer.ZipCode.apply(lambda x: len(x)).value_counts())

(1000, 17)
CustomerID         0
City               3
State              3
ZipCode            0
Country            3
Mailable           3
Emailable          3
JoinDate           0
Income           189
NetWorth           3
Education        138
Gender           198
BirthYear        215
Children           3
Churn              0
Reviews          832
Feedback_Call    838
dtype: int64
CustomerID       0.000
City             0.003
State            0.003
ZipCode          0.000
Country          0.003
Mailable         0.003
Emailable        0.003
JoinDate         0.000
Income           0.189
NetWorth         0.003
Education        0.138
Gender           0.198
BirthYear        0.215
Children         0.003
Churn            0.000
Reviews          0.832
Feedback_Call    0.838
dtype: float64
ZipCode
10    524
5     476
Name: count, dtype: int64


# Step 2: Make Your Decisions (no AI)
A decision has three parts: what you will do, an alternative you rejected, and the evidence for your choice. The first two rows are filled in as examples. **Fill in the BirthYear row yourself**, using the cell below to compute your evidence.

| Issue | Decision | Alternative rejected | Reason and evidence |
|---|---|---|---|
| Reviews and Feedback_Call have missing values | First add a flag HasReview (1 if the customer left a review), then drop both columns with the Lab 05/06 rule (keep only columns with less than 80% missing) | Fill the missing ratings with the median rating | Reviews is 83.2% and Feedback_Call 83.8% missing, so filled ratings would mostly be made up. Whether a customer left a review is real information: churn is 0.750 for reviewers and 0.719 for the others. |
| ZipCode has 5- and 9-digit values | Keep the first five digits, as in Lab 05/06 | Keep the values as they are | 476 values have five digits and 524 have nine (ZIP+4, which shows as 10 characters in Step 1 because of the hyphen). The first five digits identify the area, which is what we need. |
| BirthYear has missing values | if age is not there, fill with median |drop rows with missing values  | 21.5% of data missing |

In [51]:
# Evidence for your BirthYear decision (for example, how many are missing, and the churn rate with and without BirthYear)
# HasReview = 1 if the customer left a review (notnull() is the opposite of isnull())
customer["HasReview"] = customer["Reviews"].notnull().astype(int)
print(customer.groupby("HasReview").Churn.mean())

# add your BirthYear evidence here
customer["HasBirthYear"] = customer["BirthYear"].notnull().astype(int)
print(customer.groupby("HasBirthYear").Churn.mean())
print(customer["BirthYear"].isnull().sum())
print(customer["BirthYear"].isnull().mean())


HasReview
0    0.71875
1    0.75000
Name: Churn, dtype: float64
HasBirthYear
0    0.748837
1    0.717197
Name: Churn, dtype: float64
215
0.215


### A new feature
Feature engineering creates a new column from existing ones. Here, **AgeAtJoin** is the year of JoinDate minus BirthYear, built with `to_datetime()` and `apply()` as in Lab 05/06.

Hypothesis: older customers churn less, because they may be more settled in their choices. If so, the average AgeAtJoin should be higher for customers who returned (Churn = 0) than for customers who churned (Churn = 1). Run the cell and check.

In [52]:
customer["JoinDate"] = pd.to_datetime(customer["JoinDate"])
customer["JoinYear"] = customer.JoinDate.apply(lambda x: x.year)
customer["AgeAtJoin"] = customer["JoinYear"] - customer["BirthYear"]

# Average AgeAtJoin for returning (0) and churned (1) customers
print(customer.groupby("Churn").AgeAtJoin.mean())

Churn
0    60.918919
1    60.397869
Name: AgeAtJoin, dtype: float64


Write one or two sentences here: does the data support the hypothesis? (A hypothesis that the data does not support is still a good result if you say so clearly.)

# Step 3: Set Up the Agent's Folder
If your terminal still shows "You can now view your Streamlit app in your browser", it is busy running the app. Open a second terminal with the **+** in the terminal panel; it starts in `comm4259-work`. In that terminal, type:

```
mkdir lab_07/agent_demo
mkdir lab_07/agent_demo/data
cp lab_07/Customer.csv lab_07/agent_demo/data/
```

On Windows PowerShell, `cp` works as well. Do not upgrade Streamlit or other packages; you already noted your version in "Streamlit in Five Minutes".

# Step 4: Give the Agent Your Course Rules (CLAUDE.md)
Typing the same rules into every prompt is easy to forget. Claude Code reads a file called **`CLAUDE.md`** at the start of every session: the one in the folder where you start it, and the ones in the folders above it. A `CLAUDE.md` at the top of `comm4259-work` therefore applies to every lab and assignment folder inside it.

Run the cell below. It creates `comm4259-work/CLAUDE.md` with your Streamlit version filled in. If you already have one, it leaves yours alone and prints it.

In [53]:
from pathlib import Path
import streamlit

rules = f"""# COMM 4259: rules for Claude Code

## About me
I am a student in COMM 4259 (Foundations of Machine Learning and AI with Python). Explain things in plain language.

## Environment
- My Streamlit version is {streamlit.__version__}. Only use features available in that version.
- Do not install or upgrade any packages. If something seems to need a new package, ask me first.

## How to work
- Only create or edit files in the folder named for the current task (for example assignment_02/dashboard/ or lab_07/agent_demo/). Ask me before changing anything else.
- Follow my data-preparation decisions exactly. If one looks wrong, ask me; do not change it yourself.
- Put a short comment above each block of code that says what it does.
- When you finish, tell me how to run what you built.

"""

claude_md = Path.cwd().parent / "CLAUDE.md"   # comm4259-work/CLAUDE.md
if claude_md.exists():
    print("You already have", claude_md, "- kept as it is:\n")
else:
    claude_md.write_text(rules)
    print("Created", claude_md, "\n")
print(claude_md.read_text())

You already have /Users/sabrinali/Desktop/COMM 4259/CLAUDE.md - kept as it is:

# COMM 4259: rules for Claude Code

## About me
I am a student in COMM 4259 (Foundations of Machine Learning and AI with Python). Explain things in plain language.

## Environment
- My Streamlit version is 1.58.0. Only use features available in that version.
- Do not install or upgrade any packages. If something seems to need a new package, ask me first.

## How to work
- Only create or edit files in the folder named for the current task (for example assignment_02/dashboard/ or lab_07/agent_demo/). Ask me before changing anything else.
- Follow my data-preparation decisions exactly. If one looks wrong, ask me; do not change it yourself.
- Put a short comment above each block of code that says what it does.
- When you finish, tell me how to run what you built.



Open `comm4259-work/CLAUDE.md` in VS Code and read it. It is plain text, so you can edit it any time; add a rule of your own if you like.

**Check that Claude reads it.** In the terminal, from `comm4259-work`, type `claude`. If it asks whether you trust the folder, answer Yes: `comm4259-work` is your course folder. Then ask: *What rules are you following for this course?* It should list the rules from your file.

**Keep this Claude session open for the rest of the lab.** Do not type `/exit`. From here to Step 8 you work in this one conversation, so the agent remembers your decisions and its plan, and one `/export` at the end records everything. Whenever you need the terminal for something else (running the app, git), open another terminal with the **+** in the terminal panel. It starts in `comm4259-work`. Right-click a terminal tab and choose **Rename** to call this one `claude`.

# Step 5: Set Your Dials, Then Plan
In Lab 04 you saw the three dials of an agent session. Use them now.

1. Stay in the Claude session you started in Step 4.
2. **Model.** Type `/model` and look at the choices: a faster model for routine tasks, a deeper one for hard problems. The default is fine for this lab. Note which model you used; it goes in your AI use log.
3. **Effort.** Type `/effort` to see how hard it thinks before it acts, from low to max. A plan is worth more thought, so choose a higher level now. You can lower it later for routine fixes. Writing "ultrathink" in one prompt raises it for that prompt only.
4. **Permissions.** It asks before running a command or editing a file, and nothing happens without your yes. **Shift+Tab** switches modes, **Esc** stops it mid-step, and `/help` lists the commands. Press **Shift+Tab** until the bottom line shows **plan mode**. In plan mode the agent proposes a plan and does not change any files.
### Key idea: brief the agent like a capable new colleague
Before you paste the prompt, read how it is built. This is the Lab 04 recipe, and you will use it for every agent task this term, starting with Assignment 2. A prompt that leaves out one of these five parts is the most common reason an agent builds the wrong thing.

| Part | The question it answers | Where it is in the prompt below |
|---|---|---|
| **Goal** | What should exist at the end? | "Plan a Streamlit dashboard ... with a Data Preparation tab" |
| **Context** | What does the agent need to know that it cannot guess? | the data file, `lab_07/agent_demo/data/Customer.csv`, and your decisions, one per line |
| **Constraints** | What must it not do? | "Work only in lab_07/agent_demo/" and "Do not change my decisions; ask me if one of them looks wrong" |
| **Artifacts** | Which files should it hand back? | `app.py`, `requirements.txt`, `README.md`, an empty `ai_sessions/` folder and an empty `AI_USE_LOG.md` |
| **Finished** | How will you both know it is done? | "The tab must show" the missing values before and after, the rows and columns before and after, and the AgeAtJoin chart |

Rules that apply to every task (your Streamlit version, no new packages) are not in the prompt. They are in `CLAUDE.md`, which the agent reads at the start of every session.

5. Paste this prompt, with your BirthYear decision:

```
Work only in lab_07/agent_demo/: create and edit files only there.
Here are my data-preparation decisions for lab_07/agent_demo/data/Customer.csv:
- Reviews (83.2% missing) and Feedback_Call (83.8% missing): first add HasReview (1 if the customer left a review, else 0), then drop both columns (keep only columns with less than 80% missing).
- ZipCode (476 five-digit and 524 nine-digit values): keep the first five digits.
- BirthYear (21.5% missing): <your decision>.
New feature: AgeAtJoin = year of JoinDate minus BirthYear.
Plan a Streamlit dashboard in lab_07/agent_demo/: app.py (reading the data relative to its own file) with a Data Preparation tab (use st.tabs), requirements.txt, README.md,
an empty ai_sessions/ folder and an empty AI_USE_LOG.md (I write the log myself). The tab must show the missing values in each column before and
after my decisions, the rows and columns before and after, and a chart of the average AgeAtJoin for churned and returning customers.
Do not change my decisions; ask me if one of them looks wrong.
```

   Find each of the five parts from the table above in this prompt before you press Enter.

6. **Review the plan** before you approve it. Answer these questions in the cell below:
   - Does the plan follow each of your decisions exactly?
   - Does it follow the rules in your `CLAUDE.md` (for example, no new packages)?
   - Does it add anything you did not ask for?
   - Which files will it create?

   If any answer is not what you want, tell the agent what to change and ask for a new plan.

Your answers to the four plan-review questions:
1.
2.
3.
4.

# Step 6: Build and Run
1. Approve the plan and let the agent build. Read each permission prompt before you accept it.
2. When it finishes, type `/context` and note how full the window is (see "Watch your context" below).
3. Leave Claude running. Open a **second terminal** with the **+** in the terminal panel (it starts in `comm4259-work`) and run the app there:

```
streamlit run lab_07/agent_demo/app.py
```

(or `python -m streamlit run lab_07/agent_demo/app.py`). The app opens in your browser. This terminal is now busy running the app; the Claude terminal is still free.

4. If the app shows an error, copy the error, switch back to the Claude terminal, paste it, and ask for a fix. When it is fixed, click **Rerun** in the browser to see the fixed app.
5. Remember from Lab 04: an agent can look finished without being finished. Click through every part of the tab and compare it with your plan before you accept it.
6. Keep the app open in your browser for Step 7.

### Watch your context
Everything in a session (your prompts, the files the agent read, its replies) sits in the agent's **context window**, its working memory. The window has a limit. A long session gets slower and can lose track of early details, such as one of your decisions.

- `/context` shows how full the window is. You ran it after the build.
- `/compact` summarizes the conversation so far to free space and keeps going.
- `/clear` empties the conversation and starts fresh. `CLAUDE.md` is read again, but nothing you said earlier is remembered.

A good habit: one session per task. When you switch tasks, or `/context` shows the window filling up, start fresh.

# Step 7: Verify the Agent's Work (no AI)
Check three numbers the app shows by recomputing them yourself in the cell below. First edit the BirthYear lines so the code applies **your** decision, then run the cell and fill in the table.

In [55]:
# Apply the decisions with the Lab 05/06 commands
prepared = customer.loc[:, customer.isnull().mean() < 0.8].copy()   # drops Reviews and Feedback_Call
prepared["ZipCode"] = prepared.ZipCode.apply(lambda x: x[:5])
print("Rows and columns after the decisions:", prepared.shape)

# Your BirthYear decision: keep the one line that matches your decision, delete the others
prepared["BirthYearMissing"] = prepared.BirthYear.isnull().astype(int)                # a flag, kept with either choice
prepared["BirthYear"] = prepared.BirthYear.fillna(prepared.BirthYear.median())        # fill with the median
# prepared["BirthYear"] = prepared.BirthYear.fillna(prepared.BirthYear.mean())        # or fill with the mean
# prepared = prepared.dropna(subset=["BirthYear"])                                    # or drop those rows
print("Missing BirthYear after your decision:", prepared.BirthYear.isnull().sum())
print("Rows after your decision:", prepared.shape[0])

# The chart in the app: average AgeAtJoin for returning (0) and churned (1) customers,
# recomputed after your BirthYear decision
prepared["AgeAtJoin"] = prepared["JoinYear"] - prepared["BirthYear"]
print("Average AgeAtJoin by Churn:")
print(prepared.groupby("Churn").AgeAtJoin.mean())

Rows and columns after the decisions: (1000, 19)
Missing BirthYear after your decision: 0
Rows after your decision: 1000
Average AgeAtJoin by Churn:
Churn
0    61.043478
1    60.711326
Name: AgeAtJoin, dtype: float64


| Number | What the app shows | What your code shows | Match? |
|---|---|---|---|
| Rows and columns after the decisions | | | |
| Missing BirthYear after your decision | | | |
| Average AgeAtJoin for churned customers (Churn = 1) | | | |

If a number does not match, find out why before you move on. The agent may have applied a decision differently from what you asked.

# Step 8: Ask for a Second Option
Switch back to the Claude terminal, the same session, and ask:

```
Suggest a different reasonable way to handle BirthYear than mine, and explain the trade-off. Do not change any code.
```

Compare its option with yours, using evidence from the data (for example, how many rows each option keeps, or how it changes AgeAtJoin). Then choose one and explain why below. You may keep your original choice.

### Export the session, once, at the end
When you are done with Claude for this lab, type this in the Claude terminal:

```
/export lab_07/agent_demo/ai_sessions/lab_07.md
```

`/export` is a Claude Code command, not a prompt: with a file name after it, it writes the whole conversation to that file. The path is from `comm4259-work`, where you started Claude. Open the file in VS Code and check that it holds your prompts, the plan and the build. This is the full record of your work with the agent; a summary written by the agent does not count, so do not ask the agent to "save the conversation". Then type `/exit`.

Your comparison and choice:

# Step 9: Reflect
Write this yourself, not with the agent. Open `lab_07/agent_demo/AI_USE_LOG.md` (the agent created it empty in Step 6) and type a short entry: the tool and model you used, the effort level, your main prompts, your `/context` reading after the build, one thing the agent did that you did not ask for or got wrong, and what you changed. If you found nothing, describe one test you ran to try to break the app.

### How this lab connects to Assignment 2

| Lab 07 | Assignment 2, Section 2 |
|---|---|
| Steps 1 and 2: decide, with evidence | 2A, on the Titanic data |
| Steps 3 to 6 and the export: course rules, plan mode, build, run, `/export` | 2B |
| Steps 7 and 9: verify, reflect | 2C, check the app and report |
| Step 8: a second option | practice for weighing an AI suggestion against your own |

# Before Class Ends
Save this notebook (Cmd+S or Ctrl+S). Then, from `comm4259-work`, commit and push your work:

```
git add .
git commit -m "Lab 07: AI agent walk-through"
git push
```

# Optional: Set Up VS Code and Claude Code for Your Own Work
Two optional setups that make the rest of the term easier. Do them after class, on your own laptop. Neither is graded.

## Part 1: Helpful VS Code extensions
In VS Code, open the Extensions panel (**Cmd+Shift+X** on a Mac, **Ctrl+Shift+X** on Windows), search for each name below, check the publisher, and click **Install**.

| Extension (publisher) | What it does for you |
|---|---|
| **Claude Code** (Anthropic) | Shows the agent's edits as VS Code diffs: the old and new lines side by side, so you can review each change before you accept it. In a Claude session in the VS Code terminal, type `/ide` to connect it. |
| **Markdown Preview Enhanced** (Yiyi Wang) | Shows `.md` files formatted, with headings, tables and code blocks. Useful for your `README.md`, `AI_USE_LOG.md` and the session you exported with `/export`. Right-click a `.md` file and choose **Markdown Preview Enhanced: Open Preview to the Side**. |
| **Office Viewer** (Weijan Chen) | Opens Word, Excel and PDF files inside VS Code, for example `Data_Dictionary.xlsx` and the course PDFs, so you do not need to switch apps. |

VS Code also has a basic Markdown preview built in (**Cmd+Shift+V** or **Ctrl+Shift+V**). If a file opens in a viewer and you want to edit its text, right-click its tab and choose **Reopen Editor With... > Text Editor**.

## Part 2: Switch between two Claude accounts
Do this only if you also have a personal Claude account (for example, Claude Pro) besides your UVA account, and only on your own laptop. If you have only the UVA account, skip this part.

**Use your UVA account for all course work.** UVA's license covers university data; a personal account does not.

Claude Code keeps each account's sign-in and settings in a configuration folder. Your UVA account uses the default folder, `.claude` in your home folder. In this section you put the second account in its own folder, `.claude-personal`, and add a shortcut to your terminal's settings file. Then you choose the account when you start Claude:

```
claude --uva
claude --personal
```

Other flags still work with the shortcut, for example `claude --personal --continue`. The shortcut works only when you start Claude from a terminal; the VS Code Claude panel does not use it.

### Step A: Sign in to the second account once, in its own folder
**Mac** (the VS Code terminal):

```
CLAUDE_CONFIG_DIR="$HOME/.claude-personal" claude
```

**Windows** (the VS Code terminal, PowerShell):

```
$env:CLAUDE_CONFIG_DIR = "$HOME\.claude-personal"; claude
```

Sign in with your personal account when Claude asks, then type `/exit`. On Windows, close that terminal afterward, because the setting stays on in it.

### Step B: Add the shortcut to your terminal's settings file
**Mac.** The settings file is `~/.zshrc`. Open it with `code ~/.zshrc`, paste the lines below at the end, and save.

```
# COMM 4259: choose the Claude account when you start Claude Code
claude() {
  local want="" a
  local -a rest=()
  for a in "$@"; do
    case "$a" in
      --uva) want=uva ;;
      --personal) want=personal ;;
      *) rest+=("$a") ;;
    esac
  done
  if [[ "$want" == personal ]]; then
    CLAUDE_CONFIG_DIR="$HOME/.claude-personal" command claude "${rest[@]}"
  else
    command claude "${rest[@]}"
  fi
}
```

**Windows.** The settings file is your PowerShell profile. Create it if needed and open it:

```
if (!(Test-Path $PROFILE)) { New-Item -Path $PROFILE -ItemType File -Force }
code $PROFILE
```

Paste the lines below at the end, and save.

```
# COMM 4259: choose the Claude account when you start Claude Code
function claude {
  $rest = @(); $want = ""
  foreach ($a in $args) {
    if ($a -eq "--uva") { $want = "uva" }
    elseif ($a -eq "--personal") { $want = "personal" }
    else { $rest += $a }
  }
  $exe = (Get-Command claude -CommandType Application | Select-Object -First 1).Source
  if ($want -eq "personal") { $env:CLAUDE_CONFIG_DIR = "$HOME\.claude-personal" }
  else { Remove-Item Env:CLAUDE_CONFIG_DIR -ErrorAction SilentlyContinue }
  & $exe @rest
}
```

If PowerShell later says that running scripts is disabled, run this once and answer Yes: `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`.

If `code` is not recognized, open the file from VS Code instead (File > Open). On a Mac, press Cmd+Shift+. in the file dialog to see hidden files such as `.zshrc`.

### Step C: Check it
Open a **new** terminal, so that it reads the settings file. Then:
1. Type `claude --personal`, then `/status`. It should show your personal account. Type `/exit`.
2. Type `claude --uva` (or just `claude`), then `/status`. It should show your UVA account. Type `/exit`.